# **Models for 2018 rentals**

Goal: score a baseline and three models on the held-out test months (Oct and Nov 2018). Every score uses residual = predicted minus actual.

## **Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import joblib
import json
import pandas as pd
import sys
sys.path.append("..")

from pathlib import Path
from src.model import load_operating_rows, time_split
from src.models import (
    TARGET, SIGNED_OFF, add_flags, build_models,
    error_by, fit_and_score, group_mean_baseline, score,
)

## **Step 1: Load and split**

In [2]:
operating_rows = add_flags(load_operating_rows())
train, test = time_split(operating_rows, cutoff_date="2018-09-30")

print(f"Train rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Test mean of {TARGET}: {test[TARGET].mean():.1f}")
print(f"Months in train: {sorted(train['month'].unique())}")
print(f"Months in test: {sorted(test['month'].unique())}")

Train rows: 7,152
Test rows: 1,313
Test mean of rented_bike_count: 850.3
Months in train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(12)]
Months in test: [np.int64(10), np.int64(11)]


## **Step 2: Baselines**

A baseline predicts from the training data with no model. Every model must beat the best baseline to earn a place.

In [3]:
test_mean = test[TARGET].mean()
results = []

BASELINES = {
    "Overall mean": [],
    "Hour mean": ["hour"],
    "Hour and weekend mean": ["hour", "is_weekend"],
}
for name, keys in BASELINES.items():
    predictions = group_mean_baseline(train, test, keys)
    results.append({"model": name, **score(test[TARGET], predictions)})
res = pd.DataFrame(results)
display(res)
res[["model", "mae"]].to_string

,model,mae,rmse,r2,mean_error
0,Overall mean,452.139236,579.301171,-0.065212,-143.334452
1,Hour mean,298.485187,412.416644,0.460117,-141.139680
2,Hour and weekend mean,272.899324,378.668512,0.544859,-140.274631


<bound method DataFrame.to_string of                    model         mae
0           Overall mean  452.139236
1              Hour mean  298.485187
2  Hour and weekend mean  272.899324>

## **Step 3: Models on the signed-off features**

In [12]:
predictions_by_model = {}
for name, model in build_models(SIGNED_OFF).items():
    scores, predictions = fit_and_score(model, train, test, SIGNED_OFF)
    predictions_by_model[name] = predictions
    results.append({"model": name, **scores})

comparison = pd.DataFrame(results).sort_values("mae").reset_index(drop=True)
comparison["mae_pct_of_test_mean"] = comparison["mae"] / test_mean * 100
display(comparison.round(2))


,model,mae,rmse,r2,mean_error,mae_pct_of_test_mean
0,Gradient boosting,220.90,291.81,0.73,-187.33,25.98
1,Gradient boosting,220.90,291.81,0.73,-187.33,25.98
2,Gradient boosting,220.90,291.81,0.73,-187.33,25.98
3,Random forest,223.69,313.91,0.69,-175.04,26.31
4,Random forest,223.69,313.91,0.69,-175.04,26.31
5,Random forest,223.69,313.91,0.69,-175.04,26.31
6,Hour and weekend mean,272.90,378.67,0.54,-140.27,32.10
7,Linear regression,295.97,386.80,0.53,-212.18,34.81
8,Linear regression,295.97,386.80,0.53,-212.18,34.81
9,Linear regression,295.97,386.80,0.53,-212.18,34.81


## **Step 4: Month check**

Train holds months 12 and 1 to 9. Test holds months 10 and 11. The model has never seen a test month. This cell scores each model with and without month.

In [13]:
no_month = [f for f in SIGNED_OFF if f != "month"]
month_rows = []
for label, features in (("with month", SIGNED_OFF), ("without month", no_month)):
    for name, model in build_models(features).items():
        scores, _ = fit_and_score(model, train, test, features)
        month_rows.append({"model": name, "features": label, **scores})
display(pd.DataFrame(month_rows).round(2))

,model,features,mae,rmse,r2,mean_error
0,Linear regression,with month,295.97,386.80,0.53,-212.18
1,Random forest,with month,223.69,313.91,0.69,-175.04
2,Gradient boosting,with month,220.90,291.81,0.73,-187.33
3,Linear regression,without month,309.51,396.70,0.50,-238.68
4,Random forest,without month,262.22,348.59,0.61,-230.36
5,Gradient boosting,without month,254.08,323.58,0.67,-229.45


## **Step 5: is_weekend check**

The feature inventory left is_weekend under review. Add the flag and compare.

In [14]:
with_weekend = SIGNED_OFF + ["is_weekend"]
weekend_rows = []
for label, features in (("signed-off features", SIGNED_OFF), ("plus is_weekend", with_weekend)):
    for name, model in build_models(features).items():
        scores, _ = fit_and_score(model, train, test, features)
        weekend_rows.append({"model": name, "features": label, **scores})
display(pd.DataFrame(weekend_rows).round(2))

,model,features,mae,rmse,r2,mean_error
0,Linear regression,signed-off features,295.97,386.80,0.53,-212.18
1,Random forest,signed-off features,223.69,313.91,0.69,-175.04
2,Gradient boosting,signed-off features,220.90,291.81,0.73,-187.33
3,Linear regression,plus is_weekend,295.97,386.80,0.53,-212.18
4,Random forest,plus is_weekend,223.67,313.96,0.69,-175.10
5,Gradient boosting,plus is_weekend,220.90,291.81,0.73,-187.33


## **Step 6: Save Model and Metrics**

In [7]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

features = SIGNED_OFF
model = build_models(features)["Gradient boosting"]
scores, predictions = fit_and_score(model, train, test, features)

# Check against the figure you reported
assert abs(scores["mae"] - 220.9) < 0.05, "MAE differs from 220.9"

baseline_mae = {
    "overall_mean": score(test[TARGET], group_mean_baseline(train, test, []))["mae"],
    "hour_mean": score(test[TARGET], group_mean_baseline(train, test, ["hour"]))["mae"],
    "hour_weekend_mean": score(test[TARGET], group_mean_baseline(train, test, ["hour", "is_weekend"]))["mae"],
}

by_hour = error_by(test, predictions, "hour")

metrics = {
    "model": "Gradient boosting",
    "train_period": "December 2017 to September 2018",
    "test_period": "October and November 2018",
    "test_rows": int(len(test)),
    "features": features,
    "mae": float(scores["mae"]),
    "rmse": float(scores["rmse"]),
    "r2": float(scores["r2"]),
    "mean_error": float(scores["mean_error"]),
    "baseline_mae": {k: float(v) for k, v in baseline_mae.items()},
    "rush_hour_mean_error": {
        "08:00": float(by_hour.loc[8, "mean_error"]),
        "18:00": float(by_hour.loc[18, "mean_error"]),
    },
}

joblib.dump(model, MODELS_DIR / "gradient_boosting.joblib")
with open(MODELS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

{
  "model": "Gradient boosting",
  "train_period": "December 2017 to September 2018",
  "test_period": "October and November 2018",
  "test_rows": 1313,
  "features": [
    "hour",
    "month",
    "weekday",
    "holiday_flag",
    "temperature",
    "humidity",
    "wind_speed",
    "rainfall",
    "snowfall"
  ],
  "mae": 220.89906600146804,
  "rmse": 291.81465839432684,
  "r2": 0.7297028271030682,
  "mean_error": -187.32634771917841,
  "baseline_mae": {
    "overall_mean": 452.1392361874288,
    "hour_mean": 298.485186851158,
    "hour_weekend_mean": 272.89932385698376
  },
  "rush_hour_mean_error": {
    "08:00": -515.4466799170818,
    "18:00": -479.05786266424303
  }
}


## **Step 7: Error analysis for the best model**

Where do the misses sit? Mean error shows the direction. MAE shows the size.

In [8]:
best_name = comparison[comparison["model"].isin(predictions_by_model)].iloc[0]["model"]
best_predictions = predictions_by_model[best_name]
print(f"Best model: {best_name}")

analysis = test.copy()
analysis["rain"] = analysis["rainfall"] > 0

for column in ("month", "is_weekend", "rain", "hour"):
    print(f"\nBy {column}")
    display(error_by(analysis, best_predictions, column).round(1))

Best model: Gradient boosting

By month


,rows,mae,rmse,mean_error
month,,,,
10,665,229.1,303.9,-195.1
11,648,212.5,278.9,-179.4



By is_weekend


,rows,mae,rmse,mean_error
is_weekend,,,,
0,960,232.3,313.3,-189.3
1,353,189.9,223.2,-181.9



By rain


,rows,mae,rmse,mean_error
rain,,,,
False,1263,224.5,295.2,-196.2
True,50,129.4,186.8,37.0



By hour


,rows,mae,rmse,mean_error
hour,,,,
0,54,222.8,236.1,-207.9
1,54,186.5,207.7,-153.9
2,54,117.9,129.9,-98.2
3,54,76.8,86.4,-70.5
4,54,48.7,55.8,-33.3
5,54,42.3,50.3,-30.4
6,54,96.9,108.9,-88.0
7,55,331.4,392.8,-257.4
8,55,590.9,680.2,-515.4


## **Step 8: Findings**

- Gradient boosting scores best. Its MAE is 220.9 bikes per hour. The best baseline, the hour and weekend mean, scores 272.9. Gradient boosting beats the baseline by 52 bikes, about 19%.

- The best model leans low. Its mean error is −187.33, so predictions sit below the actual counts on average. The lean is largest at the rush hours. The mean error is −515.4 at 08:00 and −479.1 at 18:00. At the rush hours the model under-predicts by about 500 bikes.

- Adding month lowers the MAE by 15% for gradient boosting, 17% for random forest and 4% for linear regression. The result is hard to trust. The test months, October and November, never appear in the training rows. The tree models file these months with September. The gain reflects the test design and does not prove that month predicts unseen months. Two test months and one year of data add to the doubt.

- Adding is_weekend leaves the MAE unchanged: 220.9 against 220.9 for gradient boosting. Weekday already carries the same information. The flag adds nothing, so it drops out.